## Add EMEA Locations (5–8) to Simulator & Lakeflow Tables

In [ ]:
try:
    CATALOG = dbutils.widgets.get("CATALOG")
except Exception:
    CATALOG = "caspersdev"

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import random, datetime, uuid
random.seed(99)

NEW_LOCATIONS = [
    {"location_id": 5, "name": "London",    "location_code": "lon", "lat": 51.5248, "lon": -0.0796,
     "address": "14 Curtain Road, Shoreditch, London EC2A 3NH, UK", "base_orders_day": 22, "growth_rate_daily": 0.003},
    {"location_id": 6, "name": "Munich",    "location_code": "muc", "lat": 48.1601, "lon": 11.5874,
     "address": "Leopoldstrasse 75, 80802 Munich, Germany", "base_orders_day": 18, "growth_rate_daily": 0.002},
    {"location_id": 7, "name": "Amsterdam", "location_code": "ams", "lat": 52.3745, "lon": 4.8979,
     "address": "Damrak 66, 1012 LM Amsterdam, Netherlands", "base_orders_day": 20, "growth_rate_daily": 0.0025},
    {"location_id": 8, "name": "Vianen",    "location_code": "via", "lat": 51.988, "lon": 5.0895,
     "address": "Voorstraat 78, 4131 LW Vianen, Netherlands", "base_orders_day": 10, "growth_rate_daily": 0.004},
]

def col_type(table_full_name, col_name):
    """Return the Spark SQL type string for a column in an existing table."""
    return dict(spark.table(table_full_name).dtypes)[col_name]

print("Locations to add:", [l["name"] for l in NEW_LOCATIONS])

In [ ]:
# ── 1. simulator.locations ────────────────────────────────────────────────
existing_ids = {r.location_id for r in spark.table(f"{CATALOG}.simulator.locations").select("location_id").collect()}
to_add = [l for l in NEW_LOCATIONS if l["location_id"] not in existing_ids]

if to_add:
    tbl_schema = spark.table(f"{CATALOG}.simulator.locations").schema
    rows = [{c.name: l.get(c.name) for c in tbl_schema.fields} for l in to_add]
    spark.createDataFrame(rows, schema=tbl_schema).write.mode("append").saveAsTable(f"{CATALOG}.simulator.locations")
    print(f"✅ Added {len(to_add)} rows to simulator.locations")
else:
    print("ℹ️  simulator.locations already complete")

spark.table(f"{CATALOG}.simulator.locations").orderBy("location_id").show()

In [ ]:
# ── 2. simulator.brand_locations ─────────────────────────────────────────
bl_tbl = f"{CATALOG}.simulator.brand_locations"
bl = spark.table(bl_tbl)
loc_id_type = col_type(bl_tbl, "location_id")
existing_bl = {r.location_id for r in bl.select("location_id").distinct().collect()}
new_loc_ids = [l["location_id"] for l in NEW_LOCATIONS if l["location_id"] not in existing_bl]

if new_loc_ids:
    base = bl.filter(F.col("location_id") == 1)  # copy SF brand mix
    frames = [base.withColumn("location_id", F.lit(lid).cast(loc_id_type)) for lid in new_loc_ids]
    combined = frames[0]
    for df in frames[1:]:
        combined = combined.union(df)
    combined.write.mode("append").saveAsTable(bl_tbl)
    print(f"✅ Added brand assignments for location IDs: {new_loc_ids}")
else:
    print("ℹ️  brand_locations already complete")

In [ ]:
# ── 3. gold_location_sales_hourly ─────────────────────────────────────────
START_DATE = datetime.date(2024, 1, 1)
DAYS = 90
avg_price = spark.table(f"{CATALOG}.lakeflow.silver_order_items").agg(F.avg("price")).collect()[0][0] or 14.50
avg_qty = 2.5

H_WEIGHTS = [0.5]*24
for h in range(11,14): H_WEIGHTS[h] = 3.0
for h in range(17,21): H_WEIGHTS[h] = 3.5
total_w = sum(H_WEIGHTS); H_WEIGHTS = [w/total_w for w in H_WEIGHTS]

hourly_tbl = f"{CATALOG}.lakeflow.gold_location_sales_hourly"
hourly_loc_type = col_type(hourly_tbl, "location_id")
hourly_orders_type = col_type(hourly_tbl, "orders")
print(f"gold_location_sales_hourly: location_id={hourly_loc_type}, orders={hourly_orders_type}")

existing_loc_hourly = {r.location_id for r in spark.table(hourly_tbl).select("location_id").distinct().collect()}

hourly_rows = []
for loc in NEW_LOCATIONS:
    if loc["location_id"] in existing_loc_hourly:
        print(f"  skip {loc['name']} - already exists")
        continue
    for day_num in range(DAYS):
        date = START_DATE + datetime.timedelta(days=day_num)
        base_orders = loc["base_orders_day"] * ((1 + loc["growth_rate_daily"]) ** day_num)
        if date.weekday() >= 5: base_orders *= 1.3
        base_orders *= random.uniform(0.9, 1.1)
        for hour in range(24):
            hour_orders = max(0, int(base_orders * H_WEIGHTS[hour] + random.gauss(0, 0.3)))
            if hour_orders == 0: continue
            hour_ts = datetime.datetime.combine(date, datetime.time(hour, 0))
            revenue = round(hour_orders * avg_qty * avg_price * random.uniform(0.85, 1.15), 2)
            hourly_rows.append((loc["location_id"], hour_ts, hour_orders, revenue))

if hourly_rows:
    # Use SQL INSERT to avoid schema type mismatches
    tmp = "tmp_hourly_new_locs"
    schema = StructType([
        StructField("location_id", IntegerType()),
        StructField("hour_ts", TimestampType()),
        StructField("orders", LongType()),
        StructField("revenue", DoubleType()),
    ])
    tmp_df = spark.createDataFrame(hourly_rows, schema=schema)
    # Cast location_id to match target table
    tmp_df = tmp_df.withColumn("location_id", F.col("location_id").cast(hourly_loc_type)) \
                   .withColumn("orders", F.col("orders").cast(hourly_orders_type))
    tmp_df.createOrReplaceTempView(tmp)
    spark.sql(f"INSERT INTO {hourly_tbl} SELECT * FROM {tmp}")
    print(f"✅ Inserted {len(hourly_rows)} rows into gold_location_sales_hourly")
else:
    print("ℹ️  Already complete")

In [ ]:
# ── 4. gold_brand_sales_day ───────────────────────────────────────────────
brand_ids = [r.brand_id for r in spark.table(f"{CATALOG}.simulator.brands").select("brand_id").collect()]
brand_tbl = f"{CATALOG}.lakeflow.gold_brand_sales_day"
print("gold_brand_sales_day dtypes:", spark.table(brand_tbl).dtypes)

brand_day_rows = []
for loc in NEW_LOCATIONS:
    for day_num in range(DAYS):
        date = START_DATE + datetime.timedelta(days=day_num)
        base_orders = loc["base_orders_day"] * ((1 + loc["growth_rate_daily"]) ** day_num)
        if date.weekday() >= 5: base_orders *= 1.3
        base_orders *= random.uniform(0.9, 1.1)
        per_brand = max(1, int(base_orders / len(brand_ids)))
        for bid in brand_ids:
            orders = max(0, per_brand + random.randint(-1, 2))
            if orders == 0: continue
            items_sold = orders * int(avg_qty)
            brand_revenue = round(orders * avg_qty * avg_price * random.uniform(0.85, 1.15), 2)
            brand_day_rows.append((bid, date, orders, items_sold, brand_revenue))

tmp_brand = "tmp_brand_day_new_locs"
brand_schema = StructType([
    StructField("brand_id", IntegerType()), StructField("day", DateType()),
    StructField("orders", LongType()), StructField("items_sold", LongType()),
    StructField("brand_revenue", DoubleType()),
])
# Cast to match target types
target_dtypes = dict(spark.table(brand_tbl).dtypes)
tmp_df = spark.createDataFrame(brand_day_rows, schema=brand_schema)
for c, t in target_dtypes.items():
    if c in dict(tmp_df.dtypes) and dict(tmp_df.dtypes)[c] != t:
        tmp_df = tmp_df.withColumn(c, F.col(c).cast(t))
tmp_df.createOrReplaceTempView(tmp_brand)
spark.sql(f"INSERT INTO {brand_tbl} SELECT * FROM {tmp_brand}")
print(f"✅ Inserted {len(brand_day_rows)} rows into gold_brand_sales_day")

In [ ]:
# ── 5. silver_order_items + gold_order_header ─────────────────────────────
import time as _time
silver_tbl = f"{CATALOG}.lakeflow.silver_order_items"
header_tbl = f"{CATALOG}.lakeflow.gold_order_header"

# Wait up to 5 min for the lakeflow pipeline to seed at least some data for loc 1
item_sample = []
for _attempt in range(30):
    item_sample = spark.table(silver_tbl).filter(F.col("location_id") == 1).limit(200).collect()
    if item_sample:
        break
    print(f"  Waiting for silver_order_items to have data... ({_attempt+1}/30)")
    _time.sleep(10)

if not item_sample:
    print("⚠️  silver_order_items still empty — skipping silver/header seeding (run this notebook again after the lakeflow pipeline processes some events)")
    dbutils.notebook.exit("skipped: no source data yet")

silver_rows, header_rows = [], []
for loc in NEW_LOCATIONS:
    loc_id = loc["location_id"]
    for _ in range(600):
        day_num = random.randint(0, DAYS - 1)
        order_day = START_DATE + datetime.timedelta(days=day_num)
        order_id = str(uuid.uuid4())[:8].upper()
        items = random.sample(item_sample, min(random.randint(1, 4), len(item_sample)))
        order_revenue, total_qty, brand_ids_in_order = 0.0, 0, set()
        for item in items:
            qty = random.randint(1, 3)
            ext = round(item.price * qty, 2)
            order_revenue += ext; total_qty += qty; brand_ids_in_order.add(item.brand_id)
            order_ts = datetime.datetime.combine(order_day, datetime.time(random.randint(10,21), random.randint(0,59)))
            silver_rows.append((order_id, loc_id, order_ts, order_day, item.item_id, item.menu_id,
                                item.category_id, item.brand_id, item.item_name, item.price, qty, ext))
        header_rows.append((order_id, loc_id, order_day, round(order_revenue,2), total_qty, len(items), list(brand_ids_in_order)))

# silver_order_items
silver_schema = StructType([
    StructField("order_id", StringType()), StructField("location_id", IntegerType()),
    StructField("order_ts", TimestampType()), StructField("order_day", DateType()),
    StructField("item_id", IntegerType()), StructField("menu_id", IntegerType()),
    StructField("category_id", IntegerType()), StructField("brand_id", IntegerType()),
    StructField("item_name", StringType()), StructField("price", DoubleType()),
    StructField("qty", IntegerType()), StructField("extended_price", DoubleType()),
])
target_silver_dtypes = dict(spark.table(silver_tbl).dtypes)
tmp_silver = spark.createDataFrame(silver_rows, schema=silver_schema)
for c, t in target_silver_dtypes.items():
    if c in dict(tmp_silver.dtypes) and dict(tmp_silver.dtypes)[c] != t:
        tmp_silver = tmp_silver.withColumn(c, F.col(c).cast(t))
tmp_silver.createOrReplaceTempView("tmp_silver_new")
spark.sql(f"INSERT INTO {silver_tbl} SELECT * FROM tmp_silver_new")
print(f"✅ Inserted {len(silver_rows)} rows into silver_order_items")

# gold_order_header
header_schema = StructType([
    StructField("order_id", StringType()), StructField("location_id", IntegerType()),
    StructField("order_day", DateType()), StructField("order_revenue", DoubleType()),
    StructField("total_qty", LongType()), StructField("total_items", LongType()),
    StructField("brands_in_order", ArrayType(IntegerType())),
])
target_header_dtypes = dict(spark.table(header_tbl).dtypes)
tmp_header = spark.createDataFrame(header_rows, schema=header_schema)
for c, t in target_header_dtypes.items():
    if c in dict(tmp_header.dtypes) and dict(tmp_header.dtypes)[c] != t:
        tmp_header = tmp_header.withColumn(c, F.col(c).cast(t))
tmp_header.createOrReplaceTempView("tmp_header_new")
spark.sql(f"INSERT INTO {header_tbl} SELECT * FROM tmp_header_new")
print(f"✅ Inserted {len(header_rows)} rows into gold_order_header")

In [ ]:
# ── Verify ────────────────────────────────────────────────────────────────
print("Revenue by location:")
spark.table(f"{CATALOG}.lakeflow.gold_location_sales_hourly") \
    .groupBy("location_id").agg(F.round(F.sum("revenue"),2).alias("total_revenue"), F.sum("orders").alias("total_orders")) \
    .join(spark.table(f"{CATALOG}.simulator.locations").select("location_id","name"), "location_id") \
    .orderBy("location_id").show()
print("\n✅ All 8 locations now have revenue data!")